# AI Replica — Talking Face Server (SadTalker on free Colab GPU)

**Before running:** Runtime menu → Change runtime type → select **T4 GPU**.

**What this notebook does:**
1. Installs SadTalker and downloads its pretrained models
2. Lets you upload one photo of your face
3. Starts an API server that your local app sends audio to, and gets back
   a video of your face talking — with head movement and blinking, not
   just a static lip-synced mouth
4. Exposes that server publicly via ngrok

**Trade-off vs. Wav2Lip:** more natural motion, but slower — expect
roughly **1-3 minutes per reply** on a free Colab GPU, not ~10-40 seconds.
No hand or body movement (SadTalker only animates head/face).

**You need a free ngrok account** — sign up at https://ngrok.com, then
copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
and paste it into the ngrok cell below.

**Session limits:** free Colab disconnects after ~90 min idle and has a
rolling usage cap. Re-run all cells to reconnect when it drops.

In [ ]:
# 1. Install SadTalker and dependencies
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn python-multipart pyngrok nest_asyncio

# Download SadTalker's pretrained checkpoints (official download script)
!bash scripts/download_models.sh

print('Setup complete.')

In [ ]:
# 2. Upload a photo of your face (front-facing, well-lit, neutral expression works best)
from google.colab import files
import shutil

print('Upload one photo of your face:')
uploaded = files.upload()
face_filename = list(uploaded.keys())[0]
shutil.move(face_filename, 'my_face.jpg')
print('Saved as my_face.jpg')

In [ ]:
# 3. API server: accepts an audio file, runs SadTalker against my_face.jpg,
#    returns the resulting talking-head video (with head movement/blinking).
import subprocess, uuid, os, glob
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse

app = FastAPI()
os.makedirs('server_tmp', exist_ok=True)

# Set to True for higher quality face enhancement (slower). Start with False.
USE_ENHANCER = False

@app.post('/generate')
async def generate(audio: UploadFile = File(...)):
    job_id = str(uuid.uuid4())
    mp3_path = f'server_tmp/{job_id}.mp3'
    wav_path = f'server_tmp/{job_id}.wav'
    result_dir = f'server_tmp/{job_id}_result'
    os.makedirs(result_dir, exist_ok=True)

    with open(mp3_path, 'wb') as f:
        f.write(await audio.read())

    # SadTalker expects wav; convert from the mp3 our TTS produced
    subprocess.run(['ffmpeg', '-y', '-i', mp3_path, wav_path], capture_output=True)

    cmd = [
        'python', 'inference.py',
        '--driven_audio', wav_path,
        '--source_image', 'my_face.jpg',
        '--result_dir', result_dir,
        '--still',
        '--preprocess', 'full'
    ]
    if USE_ENHANCER:
        cmd += ['--enhancer', 'gfpgan']

    result = subprocess.run(cmd, capture_output=True, text=True)

    # SadTalker names its output file with a timestamp inside result_dir —
    # find whatever .mp4 it produced rather than guessing the exact name.
    mp4_files = glob.glob(f'{result_dir}/**/*.mp4', recursive=True) + glob.glob(f'{result_dir}/*.mp4')
    if not mp4_files:
        return {'error': result.stderr[-2000:]}

    return FileResponse(mp4_files[0], media_type='video/mp4')

print('API defined. Run the next cell to start it.')

In [ ]:
# 4. Start the server and expose it publicly via ngrok
from pyngrok import ngrok
import nest_asyncio, uvicorn

NGROK_AUTHTOKEN = 'PASTE_YOUR_NGROK_AUTHTOKEN_HERE'  # from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token(NGROK_AUTHTOKEN)

public_url = ngrok.connect(8000)
print('\n')
print('=========================================================')
print('COPY THIS URL into your .env file as COLAB_API_URL:')
print(public_url)
print('=========================================================')
print('\n')

nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8000)